# Relative Positionla Embeddings (RoPE)

## Summary

<img src = "../images/RoPE.png" width = "100%">

**Rotary Posisional Embeddings or RoPE** represent a paradigm shift in sequence modeling by unifying absolute and relative positional information through geometric transformations.

The mathematical brilliance of RoPE lies in the **Dot Product Linearity**. When the self-attention mechanism calculates the score between a Query ($q$) at position $i$ and a Key ($k$) at position $j$, the result depends only on the relative angle between them: $\theta_{i} - \theta_{j}$.

Because the dot product of two rotated vectors is invariant to their absolute rotation and only sensitive to their relative displacement, the model "senses" how far apart two tokens are by the degree of rotation needed to align them.

## Step by Step Explanation

### Mathematical Foundation

The fundamental objective of RoPE is to encode position $i$ by rotating the Query ($q$) and Key ($k$) vectors in a manner that preserves their relative distance.

**The Rotation Mechanism**

For a hidden dimension $d$, we treat the vector as $d/2$ pairs of coordinates. 
Assume for simplicity that our model dimension $d=4$.

$$
q = [q_1, q_2, q_3, q_4]
$$

We treat the vector as **two independent 2D planes**

* Plane 1: $(q_1, q_2)$
* Plane 2: $(q_3, q_4)$

For each pair $k \in \{1, \dots, d/2\}$, we define a position-dependent angle:

$$
\theta_{i,k} = i \cdot \Theta^{-2(k-1)/d}
$$

As shown in the example above, we calculate two unique angles ($\theta_{i,1}$ and $\theta_{i,2}$).

$$
\theta_{i,1} = i \cdot \Theta^{-2(1-1)/d} = i \\
\theta_{i,2} = i \cdot \Theta^{-2(2-1)/d} = i \cdot \Theta^{-2/d}
$$


The rotation for each pair is governed by the $2 \times 2$ matrix $R_{i,k}$:

$$
R_{i,k} = \begin{bmatrix} \cos(\theta_{i,k}) & -\sin(\theta_{i,k}) \\ \sin(\theta_{i,k}) & \cos(\theta_{i,k}) \end{bmatrix}
$$

**The Full Transformation**

These blocks are assembled into a block-diagonal matrix $R_i$, which acts on the entire embedding vector:

$$
R_i = \begin{bmatrix} 
R_{i,1} & 0 & \dots & 0 \\ 0 & R_{i,2} & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \dots & R_{i,d/2} 
\end{bmatrix}
$$

The rotated vector is computed as $q'^{(i)} = R_i q^{(i)}$. Crucially, when calculating attention between positions $i$ and $j$, the dot product satisfies:

$$
\langle R_i q^{(i)}, R_j k^{(j)} \rangle = \langle q^{(i)}, R_{j-i} k^{(j)} \rangle
$$

This demonstrates that the attention score depends solely on the relative displacement $j-i$.

### Practical Application

Let us apply this to your sequence. 

* String Sequence: `"the cat ate the rat"`
* Token Sequence: `[9, 0, 2, 7, 0, 7, 3, 9, 0, 6, 7]`
* Focus: 
    * Token 2 (the 'c' in 'cat') at Position $i=2$ vs. 
    * Token 6 (the 'r' in 'rat') at Position $i=9$.

For clarity, we will assume a small embedding dimension of $d=2$ (one rotation plane) and a base constant $\Theta = 10,000$.

#### Step 1: Calculating the Angles ($\theta$)

For $i=2$ (Token 'c'):

$$
\theta_{2} = 2 \cdot 10000^{0} = 2 \text{radians}
$$

For $i=9$ (Token 'r'):

$$
\theta_{9} = 9 \cdot 10000^{0} = 9 \text{radians}
$$

#### Step 2: Constructing the Rotation Matrices

For the 'c' token ($i=2$):

$$
R_2 = \begin{bmatrix} \cos(2) & -\sin(2) \\ \sin(2) & \cos(2) \end{bmatrix} \approx \begin{bmatrix} -0.416 & -0.909 \\ 0.909 & -0.416 \end{bmatrix}
$$

#### Step 3: Resulting Interaction

When the model performs self-attention between the Query of 'c' ($q^{(2)}$) and the Key of 'r' ($k^{(9)}$), the resulting score is influenced by the angular difference:

$$
\Delta\theta = \theta_9 - \theta_2 = 7 \text{ radians}
$$

The model "perceives" that the `'r'` in `'rat'` is exactly 7 positions ahead of the `'c'` in `'cat'`, allowing it to maintain the syntactic relationship between these subword units.

## Code

In [ ]:
"""
RoPE (Rotary Position Embedding) の実装・コメント・テスト
=========================================================

RoPE とは：
  トークンの位置情報を「回転」として埋め込む手法。
  通常の位置エンコーディングは埋め込みに足し算するが、
  RoPE は Query / Key ベクトルを位置に応じた角度だけ回転させる。

  利点：
    - 相対位置が自然に表現される（位置 m と n の内積が m-n のみに依存）
    - 長い系列への外挿がしやすい
    - LLaMA, GPT-NeoX など多くのLLMで採用されている

数学的背景：
  次元ペア (x1, x2) を角度θで回転させる操作：
    x1' = cos(θ) * x1 - sin(θ) * x2
    x2' = sin(θ) * x1 + cos(θ) * x2

  各次元ペアの回転角は位置 pos と周波数 inv_freq で決まる：
    θ_i = pos / (theta ** (2i / d_k))
    → 低次元ペアは大きな角度（速く回る）、高次元ペアは小さな角度（ゆっくり回る）
"""

import torch
import torch.nn as nn
from jaxtyping import Float
from torch import Tensor


class RoPE(nn.Module):
    """
    Rotary Position Embedding (RoPE)

    Q / K テンソルに対して位置依存の回転を適用するモジュール。
    sin / cos テーブルを事前計算してバッファにキャッシュしておくことで、
    forward のたびに再計算するコストを省く。
    """

    def __init__(
        self,
        theta: float,       # RoPE の基底周波数（論文では 10000.0 が典型）
        d_k: int,           # Q / K ベクトルの次元数（必ず偶数）
        max_seq_len: int,   # 対応できる最大系列長
        device: torch.device | None = None,
    ) -> None:

        factory_kwargs = {"device": device}

        super().__init__()

        self.theta = theta
        self.d_k = d_k
        self.max_seq_len = max_seq_len

        # sin / cos テーブルを事前計算してキャッシュ
        self._build_cache(**factory_kwargs)

    def _build_cache(self, device=None):
        """
        sin / cos のルックアップテーブルを事前計算する。

        テーブルの shape:
            sin, cos: (max_seq_len, d_k // 2)

        各 (position, dim_pair) の組み合わせについて
        θ = position * (1 / theta^(dim / d_k)) を計算し、
        その sin / cos 値を保存する。

        register_buffer を使う理由：
            - モデルの state_dict に含まれる（保存・ロードが可能）
            - model.to(device) で自動的に移動する
            - nn.Parameter ではないため勾配は計算されない
        """

        # position: (max_seq_len, 1)
        # 0, 1, 2, ..., max_seq_len-1 の列ベクトル
        position = torch.arange(
            self.max_seq_len,
            device=device
        ).unsqueeze(1)  # unsqueeze(1) で (seq_len,) → (seq_len, 1) に変形

        # dim: (d_k // 2,)
        # 0, 2, 4, ..., d_k-2 の偶数インデックス（次元ペアのインデックス）
        dim = torch.arange(
            0,
            self.d_k,
            2,          # ステップ2 → 次元ペアごとに1つの周波数
            device=device
        )

        # inv_freq: (d_k // 2,)
        # 各次元ペアの逆周波数。dim が大きいほど値が小さい（ゆっくり回る）
        inv_freq = 1.0 / (self.theta ** (dim / self.d_k))

        # sinusoid_inp: (max_seq_len, d_k // 2)
        # ブロードキャスト: (seq_len, 1) * (d_k//2,) → (seq_len, d_k//2)
        # [i, j] = position[i] * inv_freq[j]
        sinusoid_inp = position * inv_freq

        # sin, cos テーブル: (max_seq_len, d_k // 2)
        sin = torch.sin(sinusoid_inp)
        cos = torch.cos(sinusoid_inp)

        # パラメータではなくバッファとして登録
        self.register_buffer("sin", sin)
        self.register_buffer("cos", cos)

    def forward(
        self,
        x: torch.Tensor,           # (..., seq_len, d_k) — Q または K テンソル
        token_positions: torch.Tensor,  # (seq_len,) または (batch, seq_len)
    ) -> torch.Tensor:
        """
        入力テンソルに RoPE を適用して回転済みテンソルを返す。

        Args:
            x: Q または K テンソル。shape = (batch, seq_len, d_k)
            token_positions: 各トークンの位置インデックス。
                             1D (seq_len,) の場合は全バッチに同じ位置を適用。
                             KV キャッシュ使用時など位置が連続でない場合に対応。

        Returns:
            x と同 shape の回転済みテンソル。
        """

        batch_size, seq_len, dim = x.shape

        # d_k が奇数だと次元ペアを作れないのでチェック
        torch._check(
            dim % 2 == 0,
            lambda: "Embedding dimension must be even for RoPE",
        )

        # token_positions が 1D (seq_len,) の場合はバッチ次元に展開
        # expand は実際にメモリをコピーせず view を返すので効率的
        if token_positions.dim() == 1:
            token_positions = token_positions.unsqueeze(0).expand(batch_size, -1)
            # (seq_len,) → (1, seq_len) → (batch, seq_len)

        # キャッシュから該当位置の sin / cos を取り出す
        # cos, sin: (batch, seq_len, d_k // 2)
        cos = self.cos[token_positions]
        sin = self.sin[token_positions]

        # 偶数インデックス (0, 2, 4, ...) の要素 → 次元ペアの第1成分
        x1 = x[..., 0::2]  # (batch, seq_len, d_k // 2)

        # 奇数インデックス (1, 3, 5, ...) の要素 → 次元ペアの第2成分
        x2 = x[..., 1::2]  # (batch, seq_len, d_k // 2)

        # 2D 回転の適用
        # [x1']   [cos  -sin] [x1]
        # [x2'] = [sin   cos] [x2]
        real = cos * x1 - sin * x2  # 回転後の第1成分
        imag = sin * x1 + cos * x2  # 回転後の第2成分

        # 元のインターリーブ形式に戻す
        # stack で (batch, seq_len, d_k//2, 2) → flatten で (batch, seq_len, d_k)
        # stack(..., dim=-1): real と imag を最終次元で交互に並べる
        x_out = torch.stack((real, imag), dim=-1)
        x_out = x_out.flatten(-2)  # 最後の2次元 (d_k//2, 2) → (d_k,) に結合

        return x_out

In [ ]:
# ============================================================
# テストコード
# ============================================================

def main():
    print("=" * 60)
    print("RoPE テスト")
    print("=" * 60)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")

    theta       = 10000.0
    d_k         = 64
    max_seq_len = 128
    batch_size  = 4
    seq_len     = 16

    rope = RoPE(theta=theta, d_k=d_k, max_seq_len=max_seq_len, device=device)

    # --------------------------------------------------------
    # テスト A: 基本的な shape 確認
    # --------------------------------------------------------
    print("\n" + "-" * 40)
    print("テスト A: 出力 shape の確認")
    print("-" * 40)

    x = torch.randn(batch_size, seq_len, d_k, device=device)
    positions = torch.arange(seq_len, device=device)  # [0, 1, ..., seq_len-1]
    out = rope(x, positions)

    print(f"入力  shape: {x.shape}")
    print(f"位置  shape: {positions.shape}")
    print(f"出力  shape: {out.shape}  ← 入力と同じであるべき")
    assert out.shape == x.shape, "shape mismatch!"
    print("OK")

    # --------------------------------------------------------
    # テスト B: ノルムの保存確認
    # --------------------------------------------------------
    # 回転はノルムを変えない（等長変換）ので、回転前後でノルムが一致するはず
    print("\n" + "-" * 40)
    print("テスト B: ノルム保存（回転は等長変換）")
    print("-" * 40)

    norm_in  = x.norm(dim=-1)
    norm_out = out.norm(dim=-1)
    norms_match = torch.allclose(norm_in, norm_out, atol=1e-5)
    print(f"ノルム一致: {norms_match}  ({'OK' if norms_match else 'NG'})")
    print(f"最大ノルム差分: {(norm_in - norm_out).abs().max().item():.2e}")

    # --------------------------------------------------------
    # テスト C: 位置 0 では回転角 = 0 → 入力と出力が一致するはず
    # --------------------------------------------------------
    print("\n" + "-" * 40)
    print("テスト C: 位置 0 の回転は恒等変換")
    print("-" * 40)

    x_single = torch.randn(1, 1, d_k, device=device)
    pos_zero  = torch.tensor([0], device=device)
    out_zero  = rope(x_single, pos_zero)

    # position=0 → sinusoid_inp=0 → sin=0, cos=1 → 回転なし
    identity_match = torch.allclose(x_single, out_zero, atol=1e-6)
    print(f"位置0で入出力一致: {identity_match}  ({'OK' if identity_match else 'NG'})")
    print(f"最大差分: {(x_single - out_zero).abs().max().item():.2e}")

    # --------------------------------------------------------
    # テスト D: 非連続位置（KV キャッシュを想定）
    # --------------------------------------------------------
    print("\n" + "-" * 40)
    print("テスト D: 非連続な token_positions")
    print("-" * 40)

    # 例：位置 [0, 3, 7, 15] — 途中が抜けている場合
    positions_noncont = torch.tensor([0, 3, 7, 15], device=device)
    x_nc = torch.randn(batch_size, 4, d_k, device=device)
    out_nc = rope(x_nc, positions_noncont)

    print(f"非連続位置: {positions_noncont.tolist()}")
    print(f"入力  shape: {x_nc.shape}")
    print(f"出力  shape: {out_nc.shape}  ({'OK' if out_nc.shape == x_nc.shape else 'NG'})")

    # --------------------------------------------------------
    # テスト E: sin / cos キャッシュの shape 確認
    # --------------------------------------------------------
    print("\n" + "-" * 40)
    print("テスト E: sin / cos キャッシュの shape")
    print("-" * 40)

    expected_cache_shape = (max_seq_len, d_k // 2)
    print(f"sin cache shape: {rope.sin.shape}  期待値: {expected_cache_shape}")
    print(f"cos cache shape: {rope.cos.shape}  期待値: {expected_cache_shape}")
    assert rope.sin.shape == torch.Size(expected_cache_shape)
    assert rope.cos.shape == torch.Size(expected_cache_shape)
    print("OK")

    # --------------------------------------------------------
    # テスト F: 異なる位置では出力が異なる（位置情報が効いているか）
    # --------------------------------------------------------
    print("\n" + "-" * 40)
    print("テスト F: 異なる位置 → 異なる出力")
    print("-" * 40)

    x_same = torch.randn(1, 1, d_k, device=device)
    out_pos1 = rope(x_same, torch.tensor([1], device=device))
    out_pos2 = rope(x_same, torch.tensor([2], device=device))

    different = not torch.allclose(out_pos1, out_pos2)
    print(f"位置1 と 位置2 の出力が異なる: {different}  ({'OK' if different else 'NG'})")

    print("\n" + "=" * 60)
    print("すべてのテスト完了")
    print("=" * 60)


if __name__ == "__main__":
    main()